In [15]:
from pathlib import Path
import pandas as pd
import numpy as np

# Main LUAD project folder
PROJECT_DIR = Path(
    r"C:\Users\LAPTOP\Downloads\luad_reserch_code"
)

# Seven cohorts
COHORTS = [
    "TCGA_Atlas_2018",
    "LUNG_MSK_2017",
    "MSK_NPJPO_2021",
    "MSKCC_2020",
    "MSKCC_2023",
    "NCI_2022",
    "ONCOSG_2020"
]

print("Project folder exists:", PROJECT_DIR.exists())
print("Project folder:", PROJECT_DIR)
print("Number of cohorts:", len(COHORTS))
print("Cohorts:", COHORTS)

Project folder exists: True
Project folder: C:\Users\LAPTOP\Downloads\luad_reserch_code
Number of cohorts: 7
Cohorts: ['TCGA_Atlas_2018', 'LUNG_MSK_2017', 'MSK_NPJPO_2021', 'MSKCC_2020', 'MSKCC_2023', 'NCI_2022', 'ONCOSG_2020']


In [16]:
# Step 2 — Locate mutation and clinical files

REQUIRED_FILENAMES = [
    "data_mutations.txt",
    "data_clinical_patient.txt",
    "data_clinical_sample.txt"
]

located_files = {}

for filename in REQUIRED_FILENAMES:
    matches = list(PROJECT_DIR.rglob(filename))
    located_files[filename] = matches

    print(f"\n{'=' * 70}")
    print(f"FILE: {filename}")
    print(f"FOUND: {len(matches)}")
    print("=" * 70)

    if matches:
        for index, path in enumerate(matches, start=1):
            print(f"{index}. {path}")
    else:
        print("No matching file found.")


FILE: data_mutations.txt
FOUND: 0
No matching file found.

FILE: data_clinical_patient.txt
FOUND: 0
No matching file found.

FILE: data_clinical_sample.txt
FOUND: 0
No matching file found.


In [17]:
# Step 3 — Search using partial filenames

search_patterns = [
    "*mutation*",
    "*clinical*",
    "*.zip"
]

for pattern in search_patterns:
    matches = [
        path for path in PROJECT_DIR.rglob(pattern)
        if path.is_file()
    ]

    print(f"\n{'=' * 80}")
    print(f"SEARCH PATTERN: {pattern}")
    print(f"FILES FOUND: {len(matches)}")
    print("=" * 80)

    if matches:
        for index, path in enumerate(matches, start=1):
            print(f"{index}. {path}")
    else:
        print("No matching files found.")



SEARCH PATTERN: *mutation*
FILES FOUND: 68
1. C:\Users\LAPTOP\Downloads\luad_reserch_code\02_mutation_features_and_position_masks.ipynb
2. C:\Users\LAPTOP\Downloads\luad_reserch_code\05_filter_mutations.py
3. C:\Users\LAPTOP\Downloads\luad_reserch_code\lung_msk_2017_data_mutations.txt
4. C:\Users\LAPTOP\Downloads\luad_reserch_code\mskcc_2020_data_mutations.txt
5. C:\Users\LAPTOP\Downloads\luad_reserch_code\mskcc_2023_data_mutations.txt
6. C:\Users\LAPTOP\Downloads\luad_reserch_code\msk_npjpo_2021_data_mutations.txt
7. C:\Users\LAPTOP\Downloads\luad_reserch_code\nci_2022_data_mutations.txt
8. C:\Users\LAPTOP\Downloads\luad_reserch_code\oncosg_2020_data_mutations.txt
9. C:\Users\LAPTOP\Downloads\luad_reserch_code\tcga_atlas_2018_data_mutations.txt
10. C:\Users\LAPTOP\Downloads\luad_reserch_code\.venv\Lib\site-packages\sklearn\inspection\_permutation_importance.py
11. C:\Users\LAPTOP\Downloads\luad_reserch_code\.venv\Lib\site-packages\sklearn\inspection\tests\test_permutation_importance.

In [18]:
# Step 4 — Map each cohort to its exact source files

COHORT_FILE_PREFIXES = {
    "TCGA_Atlas_2018": "tcga_atlas_2018",
    "LUNG_MSK_2017": "lung_msk_2017",
    "MSK_NPJPO_2021": "msk_npjpo_2021",
    "MSKCC_2020": "mskcc_2020",
    "MSKCC_2023": "mskcc_2023",
    "NCI_2022": "nci_2022",
    "ONCOSG_2020": "oncosg_2020"
}

cohort_files = {}

for cohort, prefix in COHORT_FILE_PREFIXES.items():
    mutation_file = PROJECT_DIR / f"{prefix}_data_mutations.txt"
    patient_file = PROJECT_DIR / f"{prefix}_data_clinical_patient.txt"
    sample_file = PROJECT_DIR / f"{prefix}_data_clinical_sample.txt"

    cohort_files[cohort] = {
        "mutation": mutation_file,
        "clinical_patient": patient_file,
        "clinical_sample": sample_file
    }

    print(f"\n{cohort}")
    print("  Mutation file:", mutation_file.exists())
    print("  Patient file :", patient_file.exists())
    print("  Sample file  :", sample_file.exists())


TCGA_Atlas_2018
  Mutation file: True
  Patient file : True
  Sample file  : True

LUNG_MSK_2017
  Mutation file: True
  Patient file : True
  Sample file  : True

MSK_NPJPO_2021
  Mutation file: True
  Patient file : True
  Sample file  : True

MSKCC_2020
  Mutation file: True
  Patient file : True
  Sample file  : True

MSKCC_2023
  Mutation file: True
  Patient file : True
  Sample file  : True

NCI_2022
  Mutation file: True
  Patient file : True
  Sample file  : True

ONCOSG_2020
  Mutation file: True
  Patient file : True
  Sample file  : True


In [19]:
# Step 5 — Discover all actual column names

column_inventory = {}

for cohort, files in cohort_files.items():
    column_inventory[cohort] = {}

    print("\n" + "=" * 100)
    print("COHORT:", cohort)
    print("=" * 100)

    for file_type, file_path in files.items():
        try:
            # cBioPortal metadata rows beginning with # are ignored
            header_df = pd.read_csv(
                file_path,
                sep="\t",
                comment="#",
                nrows=0,
                encoding="utf-8"
            )

            columns = header_df.columns.tolist()
            column_inventory[cohort][file_type] = columns

            print(f"\n{file_type.upper()}")
            print(f"Number of columns: {len(columns)}")

            for column_number, column_name in enumerate(columns, start=1):
                print(f"  {column_number}. {column_name}")

        except UnicodeDecodeError:
            # Fallback for files with a different text encoding
            header_df = pd.read_csv(
                file_path,
                sep="\t",
                comment="#",
                nrows=0,
                encoding="latin-1"
            )

            columns = header_df.columns.tolist()
            column_inventory[cohort][file_type] = columns

            print(f"\n{file_type.upper()} — latin-1 encoding")
            print(f"Number of columns: {len(columns)}")

            for column_number, column_name in enumerate(columns, start=1):
                print(f"  {column_number}. {column_name}")

        except Exception as error:
            column_inventory[cohort][file_type] = []
            print(f"\n{file_type.upper()}")
            print("ERROR:", error)


COHORT: TCGA_Atlas_2018

MUTATION
Number of columns: 114
  1. Hugo_Symbol
  2. Entrez_Gene_Id
  3. Center
  4. NCBI_Build
  5. Chromosome
  6. Start_Position
  7. End_Position
  8. Strand
  9. Consequence
  10. Variant_Classification
  11. Variant_Type
  12. Reference_Allele
  13. Tumor_Seq_Allele1
  14. Tumor_Seq_Allele2
  15. dbSNP_RS
  16. dbSNP_Val_Status
  17. Tumor_Sample_Barcode
  18. Matched_Norm_Sample_Barcode
  19. Match_Norm_Seq_Allele1
  20. Match_Norm_Seq_Allele2
  21. Tumor_Validation_Allele1
  22. Tumor_Validation_Allele2
  23. Match_Norm_Validation_Allele1
  24. Match_Norm_Validation_Allele2
  25. Verification_Status
  26. Validation_Status
  27. Mutation_Status
  28. Sequencing_Phase
  29. Sequence_Source
  30. Validation_Method
  31. Score
  32. BAM_File
  33. Sequencer
  34. t_ref_count
  35. t_alt_count
  36. n_ref_count
  37. n_alt_count
  38. HGVSc
  39. HGVSp
  40. HGVSp_Short
  41. Transcript_ID
  42. RefSeq
  43. Protein_position
  44. Codons
  45. Hotspot
  4

In [20]:
# Step 6 — Map cohort-specific clinical columns to standard fields

CLINICAL_FIELD_MAP = {
    "TCGA_Atlas_2018": {
        "age": ("clinical_patient", "AGE"),
        "sex": ("clinical_patient", "SEX"),
        "smoking_status": None,
        "smoking_pack_years": None,
        "passive_smoking": None
    },

    "LUNG_MSK_2017": {
        "age": ("clinical_patient", "AGE"),
        "sex": ("clinical_patient", "SEX"),
        "smoking_status": ("clinical_patient", "SMOKING_HISTORY"),
        "smoking_pack_years": None,
        "passive_smoking": None
    },

    "MSK_NPJPO_2021": {
        "age": ("clinical_patient", "AGE_AT_RESECTION"),
        "sex": ("clinical_patient", "SEX"),
        "smoking_status": ("clinical_patient", "SMOKING_STATUS"),
        "smoking_pack_years": (
            "clinical_patient",
            "SMOKING_PACK_YEARS"
        ),
        "passive_smoking": None
    },

    "MSKCC_2020": {
        "age": ("clinical_patient", "AGE_AT_SURGERY"),
        "sex": ("clinical_patient", "SEX"),
        "smoking_status": ("clinical_patient", "SMOKING_HISTORY"),
        "smoking_pack_years": (
            "clinical_patient",
            "SMOKING_PACK_YEARS"
        ),
        "passive_smoking": None
    },

    "MSKCC_2023": {
        "age": ("clinical_sample", "AGE_AT_DOS_BX"),
        "sex": ("clinical_patient", "SEX"),
        "smoking_status": ("clinical_patient", "CIGARETTE_HX"),
        "smoking_pack_years": None,
        "passive_smoking": None
    },

    "NCI_2022": {
        "age": ("clinical_patient", "AGE"),
        "sex": ("clinical_patient", "SEX"),
        "smoking_status": None,
        "smoking_pack_years": None,
        "passive_smoking": (
            "clinical_patient",
            "PASSIVE_SMOKING"
        )
    },

    "ONCOSG_2020": {
        "age": ("clinical_patient", "AGE"),
        "sex": ("clinical_patient", "SEX"),
        "smoking_status": ("clinical_patient", "SMOKING_STATUS"),
        "smoking_pack_years": (
            "clinical_patient",
            "SMOKING_PACK_YEARS"
        ),
        "passive_smoking": None
    }
}


MUTATION_FIELD_MAP = {
    cohort: {
        "gene": "Hugo_Symbol",
        "sample_id": "Tumor_Sample_Barcode",
        "variant_classification": "Variant_Classification",
        "variant_type": "Variant_Type",
        "start_position": "Start_Position",
        "end_position": "End_Position",
        "reference_allele": "Reference_Allele",
        "alternate_allele": "Tumor_Seq_Allele2",
        "hgvs_c": "HGVSc",
        "transcript_id": "Transcript_ID"
    }
    for cohort in COHORTS
}


for cohort in COHORTS:
    print(f"\n{cohort}")

    for standard_field, mapping in CLINICAL_FIELD_MAP[cohort].items():
        if mapping is None:
            print(f"  {standard_field}: NOT AVAILABLE")
        else:
            file_type, source_column = mapping
            print(
                f"  {standard_field}: "
                f"{file_type} → {source_column}"
            )


TCGA_Atlas_2018
  age: clinical_patient → AGE
  sex: clinical_patient → SEX
  smoking_status: NOT AVAILABLE
  smoking_pack_years: NOT AVAILABLE
  passive_smoking: NOT AVAILABLE

LUNG_MSK_2017
  age: clinical_patient → AGE
  sex: clinical_patient → SEX
  smoking_status: clinical_patient → SMOKING_HISTORY
  smoking_pack_years: NOT AVAILABLE
  passive_smoking: NOT AVAILABLE

MSK_NPJPO_2021
  age: clinical_patient → AGE_AT_RESECTION
  sex: clinical_patient → SEX
  smoking_status: clinical_patient → SMOKING_STATUS
  smoking_pack_years: clinical_patient → SMOKING_PACK_YEARS
  passive_smoking: NOT AVAILABLE

MSKCC_2020
  age: clinical_patient → AGE_AT_SURGERY
  sex: clinical_patient → SEX
  smoking_status: clinical_patient → SMOKING_HISTORY
  smoking_pack_years: clinical_patient → SMOKING_PACK_YEARS
  passive_smoking: NOT AVAILABLE

MSKCC_2023
  age: clinical_sample → AGE_AT_DOS_BX
  sex: clinical_patient → SEX
  smoking_status: clinical_patient → CIGARETTE_HX
  smoking_pack_years: NOT AVAIL

In [21]:
# Step 7 — Load mapped clinical fields and inspect raw values

def read_cbioportal_file(file_path):
    """Read a cBioPortal text file while ignoring metadata lines."""
    return pd.read_csv(
        file_path,
        sep="\t",
        comment="#",
        dtype=str,
        low_memory=False,
        encoding="utf-8"
    )


def first_non_missing(series):
    """Return the first real value for patients with multiple samples."""
    cleaned = series.dropna().astype(str).str.strip()
    cleaned = cleaned[
        ~cleaned.str.lower().isin(
            ["", "nan", "na", "n/a", "null", "unknown", "not available"]
        )
    ]

    return cleaned.iloc[0] if len(cleaned) > 0 else np.nan


clinical_raw_tables = {}
clinical_audit_rows = []

for cohort in COHORTS:
    patient_df = read_cbioportal_file(
        cohort_files[cohort]["clinical_patient"]
    )

    sample_df = read_cbioportal_file(
        cohort_files[cohort]["clinical_sample"]
    )

    # Start with one row per patient
    clinical_df = patient_df[["PATIENT_ID"]].drop_duplicates().copy()
    clinical_df["COHORT"] = cohort

    for standard_field, mapping in CLINICAL_FIELD_MAP[cohort].items():

        if mapping is None:
            clinical_df[standard_field] = np.nan
            continue

        source_file, source_column = mapping

        if source_file == "clinical_patient":
            source_df = patient_df[["PATIENT_ID", source_column]].copy()
            source_df = source_df.drop_duplicates("PATIENT_ID")

        else:
            source_df = (
                sample_df[["PATIENT_ID", source_column]]
                .groupby("PATIENT_ID", as_index=False)
                .agg({source_column: first_non_missing})
            )

        source_df = source_df.rename(
            columns={source_column: standard_field}
        )

        clinical_df = clinical_df.merge(
            source_df,
            on="PATIENT_ID",
            how="left"
        )

    clinical_raw_tables[cohort] = clinical_df

    patient_count = len(clinical_df)

    def coverage_percent(column):
        return round(
            clinical_df[column].notna().mean() * 100,
            2
        )

    clinical_audit_rows.append(
        {
            "COHORT": cohort,
            "PATIENTS": patient_count,
            "AGE_AVAILABLE_%": coverage_percent("age"),
            "SEX_AVAILABLE_%": coverage_percent("sex"),
            "SMOKING_AVAILABLE_%": coverage_percent(
                "smoking_status"
            ),
            "PACK_YEARS_AVAILABLE_%": coverage_percent(
                "smoking_pack_years"
            ),
            "PASSIVE_SMOKING_AVAILABLE_%": coverage_percent(
                "passive_smoking"
            )
        }
    )

    print("\n" + "=" * 90)
    print("COHORT:", cohort)
    print("Clinical patients:", patient_count)
    print("=" * 90)

    print("\nRaw age examples:")
    print(
        clinical_df["age"]
        .dropna()
        .astype(str)
        .value_counts()
        .head(10)
    )

    print("\nRaw smoking-status values:")
    print(
        clinical_df["smoking_status"]
        .fillna("MISSING")
        .astype(str)
        .value_counts()
        .head(15)
    )

    print("\nRaw passive-smoking values:")
    print(
        clinical_df["passive_smoking"]
        .fillna("MISSING")
        .astype(str)
        .value_counts()
        .head(10)
    )


clinical_coverage_table = pd.DataFrame(clinical_audit_rows)

print("\n\nCLINICAL COVERAGE SUMMARY")
display(clinical_coverage_table)


COHORT: TCGA_Atlas_2018
Clinical patients: 566

Raw age examples:
age
70    26
72    22
60    21
59    21
61    21
69    20
71    19
74    19
65    18
67    17
Name: count, dtype: int64

Raw smoking-status values:
smoking_status
MISSING    566
Name: count, dtype: int64

Raw passive-smoking values:
passive_smoking
MISSING    566
Name: count, dtype: int64

COHORT: LUNG_MSK_2017
Clinical patients: 860

Raw age examples:
age
64    34
58    33
63    33
66    31
68    30
60    29
67    29
65    28
75    27
70    26
Name: count, dtype: int64

Raw smoking-status values:
smoking_status
Former heavy     420
Never            277
Former light     153
Current heavy     10
Name: count, dtype: int64

Raw passive-smoking values:
passive_smoking
MISSING    860
Name: count, dtype: int64

COHORT: MSK_NPJPO_2021
Clinical patients: 426

Raw age examples:
age
70    24
73    24
71    21
72    21
66    20
65    20
75    20
69    19
64    18
67    17
Name: count, dtype: int64

Raw smoking-status values:
smoki

,COHORT,PATIENTS,AGE_AVAILABLE_%,SEX_AVAILABLE_%,SMOKING_AVAILABLE_%,PACK_YEARS_AVAILABLE_%,PASSIVE_SMOKING_AVAILABLE_%
0,TCGA_Atlas_2018,566,87.46,90.81,0.00,0.00,0.00
1,LUNG_MSK_2017,860,100.00,100.00,100.00,0.00,0.00
2,MSK_NPJPO_2021,426,100.00,100.00,100.00,99.77,0.00
3,MSKCC_2020,604,100.00,100.00,100.00,99.83,0.00
4,MSKCC_2023,2201,100.00,100.00,34.80,0.00,0.00
5,NCI_2022,232,91.81,100.00,0.00,0.00,91.81
6,ONCOSG_2020,305,100.00,100.00,98.69,25.25,0.00


In [22]:
# Step 8 — Inspect exact age and smoking values before standardization

for cohort in COHORTS:
    clinical_df = clinical_raw_tables[cohort]

    print("\n" + "=" * 90)
    print("COHORT:", cohort)
    print("=" * 90)

    # Age numeric audit
    age_numeric = pd.to_numeric(
        clinical_df["age"],
        errors="coerce"
    )

    print("\nAGE SUMMARY")
    print("Non-missing raw age:", clinical_df["age"].notna().sum())
    print("Valid numeric age  :", age_numeric.notna().sum())

    if age_numeric.notna().any():
        print("Minimum age        :", age_numeric.min())
        print("Maximum age        :", age_numeric.max())
        print("Median age         :", age_numeric.median())

    print("\nSMOKING STATUS — EXACT RAW VALUES")
    smoking_values = (
        clinical_df["smoking_status"]
        .dropna()
        .astype(str)
        .str.strip()
        .value_counts(dropna=False)
    )

    if len(smoking_values) == 0:
        print("Not available")
    else:
        print(smoking_values.to_string())

    print("\nPASSIVE SMOKING — EXACT RAW VALUES")
    passive_values = (
        clinical_df["passive_smoking"]
        .dropna()
        .astype(str)
        .str.strip()
        .value_counts(dropna=False)
    )

    if len(passive_values) == 0:
        print("Not available")
    else:
        print(passive_values.to_string())


COHORT: TCGA_Atlas_2018

AGE SUMMARY
Non-missing raw age: 495
Valid numeric age  : 495
Minimum age        : 33.0
Maximum age        : 88.0
Median age         : 66.0

SMOKING STATUS — EXACT RAW VALUES
Not available

PASSIVE SMOKING — EXACT RAW VALUES
Not available

COHORT: LUNG_MSK_2017

AGE SUMMARY
Non-missing raw age: 860
Valid numeric age  : 860
Minimum age        : 9
Maximum age        : 93
Median age         : 64.0

SMOKING STATUS — EXACT RAW VALUES
smoking_status
Former heavy     420
Never            277
Former light     153
Current heavy     10

PASSIVE SMOKING — EXACT RAW VALUES
Not available

COHORT: MSK_NPJPO_2021

AGE SUMMARY
Non-missing raw age: 426
Valid numeric age  : 426
Minimum age        : 41
Maximum age        : 87
Median age         : 69.0

SMOKING STATUS — EXACT RAW VALUES
smoking_status
Ever Smoker     316
Never Smoker    110

PASSIVE SMOKING — EXACT RAW VALUES
Not available

COHORT: MSKCC_2020

AGE SUMMARY
Non-missing raw age: 604
Valid numeric age  : 604
Minimum 

In [23]:
# Step 9 — Standardize age, smoking and passive-smoking values

def standardize_smoking_detail(value):
    if pd.isna(value):
        return "Unknown"

    value = str(value).strip().lower()

    smoking_map = {
        "never": "Never",
        "never smoker": "Never",
        "no": "Never",

        "former heavy": "Former",
        "former light": "Former",

        "current heavy": "Current",

        "ever smoker": "Ever_Unspecified",
        "ever": "Ever_Unspecified",
        "yes": "Ever_Unspecified"
    }

    return smoking_map.get(value, "Unknown")


def smoking_detail_to_binary(value):
    if value == "Never":
        return "Never"

    if value in ["Former", "Current", "Ever_Unspecified"]:
        return "Ever"

    return "Unknown"


def standardize_passive_smoking(value):
    if pd.isna(value):
        return "Unknown"

    value = str(value).strip().lower()

    if value == "yes":
        return "Yes"
    elif value == "no":
        return "No"

    return "Unknown"


standardized_clinical_tables = {}

for cohort, clinical_df in clinical_raw_tables.items():
    standardized_df = clinical_df.copy()

    # Convert age to numeric but preserve missing values
    standardized_df["age_numeric"] = pd.to_numeric(
        standardized_df["age"],
        errors="coerce"
    )

    # Flag suspicious ages without deleting them
    standardized_df["age_outlier_flag"] = (
        (standardized_df["age_numeric"] < 18)
        | (standardized_df["age_numeric"] > 100)
    ).astype(int)

    standardized_df["age_missing_flag"] = (
        standardized_df["age_numeric"].isna()
    ).astype(int)

    # Standard smoking fields
    standardized_df["smoking_detail"] = (
        standardized_df["smoking_status"]
        .apply(standardize_smoking_detail)
    )

    standardized_df["smoking_binary"] = (
        standardized_df["smoking_detail"]
        .apply(smoking_detail_to_binary)
    )

    standardized_df["passive_smoking_standardized"] = (
        standardized_df["passive_smoking"]
        .apply(standardize_passive_smoking)
    )

    standardized_clinical_tables[cohort] = standardized_df

    print("\n" + "=" * 80)
    print("COHORT:", cohort)
    print("=" * 80)

    print("\nSmoking binary:")
    print(
        standardized_df["smoking_binary"]
        .value_counts(dropna=False)
        .to_string()
    )

    print("\nSmoking detail:")
    print(
        standardized_df["smoking_detail"]
        .value_counts(dropna=False)
        .to_string()
    )

    print("\nAge missing:", standardized_df["age_missing_flag"].sum())
    print("Age outliers:", standardized_df["age_outlier_flag"].sum())

    if standardized_df["age_outlier_flag"].sum() > 0:
        print("\nSuspicious age records:")
        print(
            standardized_df.loc[
                standardized_df["age_outlier_flag"] == 1,
                ["PATIENT_ID", "age_numeric"]
            ].to_string(index=False)
        )


COHORT: TCGA_Atlas_2018

Smoking binary:
smoking_binary
Unknown    566

Smoking detail:
smoking_detail
Unknown    566

Age missing: 71
Age outliers: 0

COHORT: LUNG_MSK_2017

Smoking binary:
smoking_binary
Ever     583
Never    277

Smoking detail:
smoking_detail
Former     573
Never      277
Current     10

Age missing: 0
Age outliers: 2

Suspicious age records:
PATIENT_ID  age_numeric
 P-0005738            9
 P-0009339           16

COHORT: MSK_NPJPO_2021

Smoking binary:
smoking_binary
Ever     316
Never    110

Smoking detail:
smoking_detail
Ever_Unspecified    316
Never               110

Age missing: 0
Age outliers: 0

COHORT: MSKCC_2020

Smoking binary:
smoking_binary
Ever     466
Never    138

Smoking detail:
smoking_detail
Ever_Unspecified    466
Never               138

Age missing: 0
Age outliers: 0

COHORT: MSKCC_2023

Smoking binary:
smoking_binary
Unknown    1435
Ever        550
Never       216

Smoking detail:
smoking_detail
Unknown             1435
Ever_Unspecified    

In [24]:
# Step 10 — Combine standardized clinical data from all cohorts

combined_clinical_parts = []

for cohort, clinical_df in standardized_clinical_tables.items():
    cohort_df = clinical_df.copy()

    # Use plausible adult age range for modelling
    # Raw age and outlier flag are still preserved
    cohort_df["age_for_model"] = cohort_df["age_numeric"].mask(
        cohort_df["age_outlier_flag"] == 1
    )

    # Convert pack-years to numeric where available
    cohort_df["smoking_pack_years_numeric"] = pd.to_numeric(
        cohort_df["smoking_pack_years"],
        errors="coerce"
    )

    # Negative pack-year values are invalid
    cohort_df.loc[
        cohort_df["smoking_pack_years_numeric"] < 0,
        "smoking_pack_years_numeric"
    ] = np.nan

    cohort_df["pack_years_missing_flag"] = (
        cohort_df["smoking_pack_years_numeric"].isna()
    ).astype(int)

    combined_clinical_parts.append(
        cohort_df[
            [
                "COHORT",
                "PATIENT_ID",
                "age",
                "age_numeric",
                "age_for_model",
                "age_missing_flag",
                "age_outlier_flag",
                "sex",
                "smoking_status",
                "smoking_detail",
                "smoking_binary",
                "smoking_pack_years",
                "smoking_pack_years_numeric",
                "pack_years_missing_flag",
                "passive_smoking",
                "passive_smoking_standardized"
            ]
        ]
    )


combined_clinical_df = pd.concat(
    combined_clinical_parts,
    ignore_index=True
)

duplicate_count = combined_clinical_df.duplicated(
    subset=["COHORT", "PATIENT_ID"]
).sum()

clinical_combined_summary = (
    combined_clinical_df
    .groupby("COHORT")
    .agg(
        PATIENTS=("PATIENT_ID", "size"),
        AGE_USABLE_PERCENT=(
            "age_for_model",
            lambda x: round(x.notna().mean() * 100, 2)
        ),
        AGE_OUTLIERS=("age_outlier_flag", "sum"),
        SMOKING_KNOWN_PERCENT=(
            "smoking_binary",
            lambda x: round((x != "Unknown").mean() * 100, 2)
        ),
        PACK_YEARS_USABLE_PERCENT=(
            "smoking_pack_years_numeric",
            lambda x: round(x.notna().mean() * 100, 2)
        ),
        PASSIVE_SMOKING_KNOWN_PERCENT=(
            "passive_smoking_standardized",
            lambda x: round((x != "Unknown").mean() * 100, 2)
        )
    )
    .reset_index()
)

print("Combined clinical shape:", combined_clinical_df.shape)
print(
    "Duplicate COHORT + PATIENT_ID records:",
    duplicate_count
)

print("\nCombined clinical coverage:")
display(clinical_combined_summary)

Combined clinical shape: (5194, 16)
Duplicate COHORT + PATIENT_ID records: 0

Combined clinical coverage:


,COHORT,PATIENTS,AGE_USABLE_PERCENT,AGE_OUTLIERS,SMOKING_KNOWN_PERCENT,PACK_YEARS_USABLE_PERCENT,PASSIVE_SMOKING_KNOWN_PERCENT
0,LUNG_MSK_2017,860,99.77,2,100.00,0.00,0.00
1,MSKCC_2020,604,100.00,0,100.00,99.83,0.00
2,MSKCC_2023,2201,100.00,0,34.80,0.00,0.00
3,MSK_NPJPO_2021,426,100.00,0,100.00,99.77,0.00
4,NCI_2022,232,91.81,0,0.00,0.00,91.81
5,ONCOSG_2020,305,100.00,0,98.69,25.25,0.00
6,TCGA_Atlas_2018,566,87.46,0,0.00,0.00,0.00


In [25]:
# Step 11 — Merge clinical covariates with final LUAD patient dataset

FINAL_PATIENT_DATASET_PATH = (
    PROJECT_DIR
    / "luad_processed"
    / "final_luad_minion_patient_dataset.csv"
)

final_patient_df = pd.read_csv(
    FINAL_PATIENT_DATASET_PATH,
    low_memory=False
)

# Convert source cohort names to the cohort names used
# inside the final patient-level dataset
SOURCE_TO_FINAL_COHORT = {
    "TCGA_Atlas_2018": "TCGA",
    "LUNG_MSK_2017": "LUNG",
    "MSK_NPJPO_2021": "MSK_NPJPO",
    "MSKCC_2020": "MSKCC2020",
    "MSKCC_2023": "MSKCC2023",
    "NCI_2022": "NCI",
    "ONCOSG_2020": "ONCOSG"
}

clinical_for_merge = combined_clinical_df.copy()

clinical_for_merge["COHORT"] = (
    clinical_for_merge["COHORT"]
    .map(SOURCE_TO_FINAL_COHORT)
)

clinical_for_merge = clinical_for_merge.rename(
    columns={"sex": "clinical_sex"}
)

clinical_columns_to_keep = [
    "COHORT",
    "PATIENT_ID",
    "age_numeric",
    "age_for_model",
    "age_missing_flag",
    "age_outlier_flag",
    "clinical_sex",
    "smoking_detail",
    "smoking_binary",
    "smoking_pack_years_numeric",
    "pack_years_missing_flag",
    "passive_smoking_standardized"
]

clinical_for_merge = clinical_for_merge[
    clinical_columns_to_keep
].copy()

# Merge patient-level clinical features
enhanced_patient_df = final_patient_df.merge(
    clinical_for_merge,
    on=["COHORT", "PATIENT_ID"],
    how="left",
    validate="one_to_one",
    indicator=True
)

enhanced_patient_df["clinical_record_matched"] = (
    enhanced_patient_df["_merge"] == "both"
)

enhanced_patient_df["smoking_known"] = (
    enhanced_patient_df["smoking_binary"].notna()
    & enhanced_patient_df["smoking_binary"].ne("Unknown")
)

enhanced_patient_df["passive_smoking_known"] = (
    enhanced_patient_df[
        "passive_smoking_standardized"
    ].notna()
    & enhanced_patient_df[
        "passive_smoking_standardized"
    ].ne("Unknown")
)

matched_coverage = (
    enhanced_patient_df
    .groupby("COHORT")
    .agg(
        FINAL_PATIENTS=("PATIENT_ID", "size"),

        CLINICAL_MATCH_PERCENT=(
            "clinical_record_matched",
            lambda x: round(x.mean() * 100, 2)
        ),

        AGE_USABLE_PERCENT=(
            "age_for_model",
            lambda x: round(x.notna().mean() * 100, 2)
        ),

        SMOKING_KNOWN_PERCENT=(
            "smoking_known",
            lambda x: round(x.mean() * 100, 2)
        ),

        PACK_YEARS_USABLE_PERCENT=(
            "smoking_pack_years_numeric",
            lambda x: round(x.notna().mean() * 100, 2)
        ),

        PASSIVE_SMOKING_KNOWN_PERCENT=(
            "passive_smoking_known",
            lambda x: round(x.mean() * 100, 2)
        )
    )
    .reset_index()
)

print("Original final dataset shape:", final_patient_df.shape)
print("Enhanced merged shape:", enhanced_patient_df.shape)

print(
    "Matched clinical records:",
    int(enhanced_patient_df["clinical_record_matched"].sum()),
    "/",
    len(enhanced_patient_df)
)

print(
    "Unmatched patients:",
    int((~enhanced_patient_df["clinical_record_matched"]).sum())
)

print("\nActual coverage in final training population:")
display(matched_coverage)

Original final dataset shape: (3382, 13)
Enhanced merged shape: (3382, 27)
Matched clinical records: 0 / 3382
Unmatched patients: 3382

Actual coverage in final training population:


,COHORT,FINAL_PATIENTS,CLINICAL_MATCH_PERCENT,AGE_USABLE_PERCENT,SMOKING_KNOWN_PERCENT,PACK_YEARS_USABLE_PERCENT,PASSIVE_SMOKING_KNOWN_PERCENT
0,LUNG_MSK_2017,779,0.0,0.0,0.0,0.0,0.0
1,MSKCC_2020,555,0.0,0.0,0.0,0.0,0.0
2,MSKCC_2023,745,0.0,0.0,0.0,0.0,0.0
3,MSK_NPJPO_2021,396,0.0,0.0,0.0,0.0,0.0
4,NCI_2022,151,0.0,0.0,0.0,0.0,0.0
5,ONCOSG_2020,268,0.0,0.0,0.0,0.0,0.0
6,TCGA_Atlas_2018,488,0.0,0.0,0.0,0.0,0.0


In [26]:
# Step 11 corrected — Merge without changing cohort names

final_patient_df = pd.read_csv(
    FINAL_PATIENT_DATASET_PATH,
    low_memory=False
)

clinical_for_merge = combined_clinical_df.copy()

# Clean merge keys on both sides
final_patient_df["COHORT"] = (
    final_patient_df["COHORT"].astype(str).str.strip()
)

final_patient_df["PATIENT_ID"] = (
    final_patient_df["PATIENT_ID"].astype(str).str.strip()
)

clinical_for_merge["COHORT"] = (
    clinical_for_merge["COHORT"].astype(str).str.strip()
)

clinical_for_merge["PATIENT_ID"] = (
    clinical_for_merge["PATIENT_ID"].astype(str).str.strip()
)

clinical_for_merge = clinical_for_merge.rename(
    columns={"sex": "clinical_sex"}
)

clinical_for_merge = clinical_for_merge[
    [
        "COHORT",
        "PATIENT_ID",
        "age_numeric",
        "age_for_model",
        "age_missing_flag",
        "age_outlier_flag",
        "clinical_sex",
        "smoking_detail",
        "smoking_binary",
        "smoking_pack_years_numeric",
        "pack_years_missing_flag",
        "passive_smoking_standardized"
    ]
].copy()

enhanced_patient_df = final_patient_df.merge(
    clinical_for_merge,
    on=["COHORT", "PATIENT_ID"],
    how="left",
    validate="one_to_one",
    indicator=True
)

enhanced_patient_df["clinical_record_matched"] = (
    enhanced_patient_df["_merge"] == "both"
)

enhanced_patient_df["smoking_known"] = (
    enhanced_patient_df["smoking_binary"].notna()
    & enhanced_patient_df["smoking_binary"].ne("Unknown")
)

enhanced_patient_df["passive_smoking_known"] = (
    enhanced_patient_df["passive_smoking_standardized"].notna()
    & enhanced_patient_df[
        "passive_smoking_standardized"
    ].ne("Unknown")
)

matched_coverage = (
    enhanced_patient_df
    .groupby("COHORT")
    .agg(
        FINAL_PATIENTS=("PATIENT_ID", "size"),

        CLINICAL_MATCH_PERCENT=(
            "clinical_record_matched",
            lambda x: round(x.mean() * 100, 2)
        ),

        AGE_USABLE_PERCENT=(
            "age_for_model",
            lambda x: round(x.notna().mean() * 100, 2)
        ),

        SMOKING_KNOWN_PERCENT=(
            "smoking_known",
            lambda x: round(x.mean() * 100, 2)
        ),

        PACK_YEARS_USABLE_PERCENT=(
            "smoking_pack_years_numeric",
            lambda x: round(x.notna().mean() * 100, 2)
        ),

        PASSIVE_SMOKING_KNOWN_PERCENT=(
            "passive_smoking_known",
            lambda x: round(x.mean() * 100, 2)
        )
    )
    .reset_index()
)

print("Original final dataset shape:", final_patient_df.shape)
print("Enhanced merged shape:", enhanced_patient_df.shape)

print(
    "Matched clinical records:",
    int(enhanced_patient_df["clinical_record_matched"].sum()),
    "/",
    len(enhanced_patient_df)
)

print(
    "Unmatched patients:",
    int((~enhanced_patient_df["clinical_record_matched"]).sum())
)

print("\nActual coverage in final training population:")
display(matched_coverage)

Original final dataset shape: (3382, 13)
Enhanced merged shape: (3382, 27)
Matched clinical records: 3382 / 3382
Unmatched patients: 0

Actual coverage in final training population:


,COHORT,FINAL_PATIENTS,CLINICAL_MATCH_PERCENT,AGE_USABLE_PERCENT,SMOKING_KNOWN_PERCENT,PACK_YEARS_USABLE_PERCENT,PASSIVE_SMOKING_KNOWN_PERCENT
0,LUNG_MSK_2017,779,100.0,99.74,100.00,0.00,0.0
1,MSKCC_2020,555,100.0,100.00,100.00,99.82,0.0
2,MSKCC_2023,745,100.0,100.00,95.97,0.00,0.0
3,MSK_NPJPO_2021,396,100.0,100.00,100.00,99.75,0.0
4,NCI_2022,151,100.0,92.05,0.00,0.00,94.7
5,ONCOSG_2020,268,100.0,100.00,98.51,24.25,0.0
6,TCGA_Atlas_2018,488,100.0,96.11,0.00,0.00,0.0


In [27]:
# Step 12 — Validate clinical merge and sex consistency

def normalize_sex_value(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    if value in ["female", "f"]:
        return "Female"

    if value in ["male", "m"]:
        return "Male"

    return np.nan


enhanced_patient_df["original_sex_standardized"] = (
    enhanced_patient_df["SEX"].apply(normalize_sex_value)
)

enhanced_patient_df["clinical_sex_standardized"] = (
    enhanced_patient_df["clinical_sex"].apply(normalize_sex_value)
)

sex_comparable = (
    enhanced_patient_df["original_sex_standardized"].notna()
    & enhanced_patient_df["clinical_sex_standardized"].notna()
)

sex_mismatch = (
    sex_comparable
    & (
        enhanced_patient_df["original_sex_standardized"]
        != enhanced_patient_df["clinical_sex_standardized"]
    )
)

overall_clinical_summary = pd.DataFrame(
    {
        "Check": [
            "Total final patients",
            "Clinical records matched",
            "Usable age",
            "Known active smoking",
            "Known passive smoking",
            "Comparable sex records",
            "Sex mismatches",
            "Duplicate cohort-patient IDs"
        ],
        "Value": [
            len(enhanced_patient_df),
            int(enhanced_patient_df["clinical_record_matched"].sum()),
            int(enhanced_patient_df["age_for_model"].notna().sum()),
            int(enhanced_patient_df["smoking_known"].sum()),
            int(enhanced_patient_df["passive_smoking_known"].sum()),
            int(sex_comparable.sum()),
            int(sex_mismatch.sum()),
            int(
                enhanced_patient_df.duplicated(
                    subset=["COHORT", "PATIENT_ID"]
                ).sum()
            )
        ]
    }
)

print("Final clinical sanity-check summary:")
display(overall_clinical_summary)

print("\nOverall usable percentages:")
print(
    "Age:",
    round(
        enhanced_patient_df["age_for_model"].notna().mean() * 100,
        2
    ),
    "%"
)

print(
    "Active smoking:",
    round(
        enhanced_patient_df["smoking_known"].mean() * 100,
        2
    ),
    "%"
)

if sex_mismatch.any():
    print("\nSex mismatch records:")
    display(
        enhanced_patient_df.loc[
            sex_mismatch,
            [
                "COHORT",
                "PATIENT_ID",
                "SEX",
                "clinical_sex"
            ]
        ]
    )
else:
    print("\nSex validation passed: no mismatches found.")

Final clinical sanity-check summary:


,Check,Value
0,Total final patients,3382
1,Clinical records matched,3382
2,Usable age,3349
3,Known active smoking,2709
4,Known passive smoking,143
5,Comparable sex records,3382
6,Sex mismatches,0
7,Duplicate cohort-patient IDs,0



Overall usable percentages:
Age: 99.02 %
Active smoking: 80.1 %

Sex validation passed: no mismatches found.


In [30]:
# Step 13 — Save final clinical covariates and TXT summary
# Running this cell again overwrites the same two files.
# Passive smoking is completely excluded.

from pathlib import Path
import pandas as pd

IMPROVED_OUTPUT_DIR = (
    PROJECT_DIR
    / "luad_processeed_data"
)

IMPROVED_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLINICAL_OUTPUT_PATH = (
    IMPROVED_OUTPUT_DIR
    / "luad_improved_clinical_covariates.csv"
)

SUMMARY_TXT_PATH = (
    IMPROVED_OUTPUT_DIR
    / "luad_improved_clinical_summary.txt"
)

# Remove old CSV summary created by the previous code
OLD_SUMMARY_CSV_PATH = (
    IMPROVED_OUTPUT_DIR
    / "luad_improved_clinical_summary.csv"
)

if OLD_SUMMARY_CSV_PATH.exists():
    OLD_SUMMARY_CSV_PATH.unlink()
    print("Removed old summary CSV:", OLD_SUMMARY_CSV_PATH)


# ---------------------------------------------------------
# Prepare final clinical dataset
# ---------------------------------------------------------

clinical_output_df = enhanced_patient_df.copy()

clinical_output_df["age_unusable_flag"] = (
    clinical_output_df["age_for_model"].isna()
).astype(int)

clinical_output_df["smoking_unknown_flag"] = (
    clinical_output_df["smoking_binary"].isna()
    | clinical_output_df["smoking_binary"].eq("Unknown")
).astype(int)

# Passive-smoking columns are not selected
clinical_output_df = clinical_output_df[
    [
        "COHORT",
        "SAMPLE_ID",
        "PATIENT_ID",

        "SEX",
        "sex_encoded",

        "age_numeric",
        "age_for_model",
        "age_unusable_flag",
        "age_outlier_flag",

        "smoking_detail",
        "smoking_binary",
        "smoking_unknown_flag",

        "smoking_pack_years_numeric",
        "pack_years_missing_flag",

        "clinical_record_matched"
    ]
].copy()

clinical_output_df = clinical_output_df.sort_values(
    ["COHORT", "PATIENT_ID"]
).reset_index(drop=True)


# ---------------------------------------------------------
# Save CSV — overwrite existing file
# ---------------------------------------------------------

clinical_output_df.to_csv(
    CLINICAL_OUTPUT_PATH,
    index=False,
    mode="w"
)


# ---------------------------------------------------------
# Calculate overall summary
# ---------------------------------------------------------

total_patients = len(clinical_output_df)

usable_age_count = int(
    clinical_output_df["age_for_model"].notna().sum()
)

known_smoking_count = int(
    clinical_output_df["smoking_binary"]
    .isin(["Never", "Ever"])
    .sum()
)

unknown_smoking_count = int(
    clinical_output_df["smoking_unknown_flag"].sum()
)

age_outlier_count = int(
    clinical_output_df["age_outlier_flag"].sum()
)

duplicate_count = int(
    clinical_output_df.duplicated(
        subset=["COHORT", "PATIENT_ID"]
    ).sum()
)

clinical_match_count = int(
    clinical_output_df["clinical_record_matched"].sum()
)

passive_columns = [
    column
    for column in clinical_output_df.columns
    if "passive" in column.lower()
]


# ---------------------------------------------------------
# Create cohort-wise summary
# ---------------------------------------------------------

cohort_summary = (
    clinical_output_df
    .groupby("COHORT")
    .agg(
        total_patients=("PATIENT_ID", "size"),

        usable_age_count=(
            "age_for_model",
            lambda x: int(x.notna().sum())
        ),

        known_smoking_count=(
            "smoking_binary",
            lambda x: int(x.isin(["Never", "Ever"]).sum())
        ),

        unknown_smoking_count=(
            "smoking_unknown_flag",
            "sum"
        ),

        age_outlier_count=(
            "age_outlier_flag",
            "sum"
        )
    )
    .reset_index()
)

cohort_summary["usable_age_percent"] = (
    cohort_summary["usable_age_count"]
    / cohort_summary["total_patients"]
    * 100
).round(2)

cohort_summary["known_smoking_percent"] = (
    cohort_summary["known_smoking_count"]
    / cohort_summary["total_patients"]
    * 100
).round(2)


# ---------------------------------------------------------
# Write TXT summary — overwrite existing file
# ---------------------------------------------------------

summary_lines = [
    "LUAD IMPROVED CLINICAL COVARIATES SUMMARY",
    "=" * 60,
    "",
    f"Total patients: {total_patients}",
    f"Matched clinical records: {clinical_match_count}",
    f"Duplicate COHORT + PATIENT_ID records: {duplicate_count}",
    "",
    f"Usable age records: {usable_age_count}",
    f"Usable age percentage: "
    f"{usable_age_count / total_patients * 100:.2f}%",
    f"Age outlier records: {age_outlier_count}",
    "",
    f"Known smoking records: {known_smoking_count}",
    f"Known smoking percentage: "
    f"{known_smoking_count / total_patients * 100:.2f}%",
    f"Unknown smoking records: {unknown_smoking_count}",
    "",
    "Passive smoking included: No",
    f"Passive smoking columns found: {passive_columns}",
    "",
    "PACK-YEARS NOTE:",
    "Pack-years were preserved for auditing and optional experiments.",
    "They will not be used in the main model because coverage is inconsistent.",
    "",
    "COHORT-WISE SUMMARY",
    "=" * 60
]

for _, row in cohort_summary.iterrows():
    summary_lines.extend(
        [
            "",
            f"Cohort: {row['COHORT']}",
            f"  Total patients: {int(row['total_patients'])}",
            f"  Usable age: {int(row['usable_age_count'])} "
            f"({row['usable_age_percent']:.2f}%)",
            f"  Known smoking: {int(row['known_smoking_count'])} "
            f"({row['known_smoking_percent']:.2f}%)",
            f"  Unknown smoking: "
            f"{int(row['unknown_smoking_count'])}",
            f"  Age outliers: {int(row['age_outlier_count'])}"
        ]
    )

with open(
    SUMMARY_TXT_PATH,
    mode="w",
    encoding="utf-8"
) as summary_file:
    summary_file.write("\n".join(summary_lines))


# ---------------------------------------------------------
# Final verification
# ---------------------------------------------------------

print("\nClinical CSV saved:")
print(CLINICAL_OUTPUT_PATH)

print("\nTXT summary saved:")
print(SUMMARY_TXT_PATH)

print("\nCSV shape:", clinical_output_df.shape)
print("Duplicate patients:", duplicate_count)
print("Passive smoking columns:", passive_columns)

print("\nFiles currently created by this cell:")
print("1.", CLINICAL_OUTPUT_PATH.name)
print("2.", SUMMARY_TXT_PATH.name)


Clinical CSV saved:
C:\Users\LAPTOP\Downloads\luad_reserch_code\luad_processeed_data\luad_improved_clinical_covariates.csv

TXT summary saved:
C:\Users\LAPTOP\Downloads\luad_reserch_code\luad_processeed_data\luad_improved_clinical_summary.txt

CSV shape: (3382, 15)
Duplicate patients: 0
Passive smoking columns: []

Files currently created by this cell:
1. luad_improved_clinical_covariates.csv
2. luad_improved_clinical_summary.txt


In [31]:
# Step 14 — Compare stage distribution in smoking-missing cohorts

SMOKING_UNAVAILABLE_COHORTS = [
    "NCI_2022",
    "TCGA_Atlas_2018"
]

stage_count_by_cohort = pd.crosstab(
    enhanced_patient_df["COHORT"],
    enhanced_patient_df["STAGE_BINARY"]
)

stage_percent_by_cohort = (
    pd.crosstab(
        enhanced_patient_df["COHORT"],
        enhanced_patient_df["STAGE_BINARY"],
        normalize="index"
    )
    * 100
).round(2)

print("Stage counts by cohort:")
display(stage_count_by_cohort)

print("\nStage percentages by cohort:")
display(stage_percent_by_cohort)


# Compare cohorts with and without smoking data
stage_comparison_df = enhanced_patient_df[
    ["COHORT", "STAGE_BINARY"]
].copy()

stage_comparison_df["SMOKING_COVERAGE_GROUP"] = np.where(
    stage_comparison_df["COHORT"].isin(
        SMOKING_UNAVAILABLE_COHORTS
    ),
    "Smoking completely unavailable",
    "Smoking partly/fully available"
)

group_stage_counts = pd.crosstab(
    stage_comparison_df["SMOKING_COVERAGE_GROUP"],
    stage_comparison_df["STAGE_BINARY"]
)

group_stage_percentages = (
    pd.crosstab(
        stage_comparison_df["SMOKING_COVERAGE_GROUP"],
        stage_comparison_df["STAGE_BINARY"],
        normalize="index"
    )
    * 100
).round(2)

print("\nGrouped stage counts:")
display(group_stage_counts)

print("\nGrouped stage percentages:")
display(group_stage_percentages)

Stage counts by cohort:


STAGE_BINARY,Advanced,Early
COHORT,,
LUNG_MSK_2017,695,84
MSKCC_2020,58,497
MSKCC_2023,302,443
MSK_NPJPO_2021,0,396
NCI_2022,30,121
ONCOSG_2020,94,174
TCGA_Atlas_2018,107,381



Stage percentages by cohort:


STAGE_BINARY,Advanced,Early
COHORT,,
LUNG_MSK_2017,89.22,10.78
MSKCC_2020,10.45,89.55
MSKCC_2023,40.54,59.46
MSK_NPJPO_2021,0.00,100.00
NCI_2022,19.87,80.13
ONCOSG_2020,35.07,64.93
TCGA_Atlas_2018,21.93,78.07



Grouped stage counts:


STAGE_BINARY,Advanced,Early
SMOKING_COVERAGE_GROUP,,
Smoking completely unavailable,137,502
Smoking partly/fully available,1149,1594



Grouped stage percentages:


STAGE_BINARY,Advanced,Early
SMOKING_COVERAGE_GROUP,,
Smoking completely unavailable,21.44,78.56
Smoking partly/fully available,41.89,58.11
